# Notebook 08 — Persistence Validation**Luke Wardle. Branch:** `feat/persistence-validation`> **NOTE ON NUMBERING.** Notebook 07 reserved 08 for classifier training. This> notebook takes that slot deliberately: it tests whether the candidate pool a> classifier would be trained on contains the thing we intend to sell. That> question is upstream of any model, and the answer here determines whether the> classifier is worth building. Model training moves to Notebook 09.## The questionThe product is defined as **persistently-bare, unregistered, developable land**.Every word of that definition except *persistently* has been tested. Persistencehad never been measured, because until now the pipeline had only ever been runagainst a single image date, and a single date cannot distinguish land that isbare because it is derelict from land that is bare because it was ploughed,stripped or resurfaced the week before.This notebook runs the pipeline across four seasonal scenes spanning fourteenmonths, isolates the candidates that are bare in **all four**, and inspects themagainst aerial imagery.The hypothesis under test:> Requiring a candidate to be bare across four seasons removes transient bare> ground and leaves a set enriched in genuine derelict, developable land.The hypothesis is falsifiable in an obvious way: if the persistent set is notenriched in sellable land, persistence is not the filter the product needs.

## 1. Method — choosing the scenesFour scenes were selected manually in Copernicus Browser, one per season, allSentinel-2 L2A over Stoke-on-Trent (GSS E06000021).**Scene selection could not be automated, and the reason is worth recording.**The OData `cloudCover` attribute the pipeline filters on is measured across theentire Sentinel-2 tile — roughly 110 x 110 km. Stoke occupies a small fraction ofthat footprint. A scene can therefore report 8% cloud tile-wide while a cloudbank sits directly over the city, or report 25% while Stoke itself is perfectlyclear beneath an otherwise cloudy tile. Filtering on the reported percentageselects the wrong scenes in both directions.Each candidate date was instead inspected visually in true colour, judging cloud**over the council area specifically** and disregarding cloud elsewhere in thetile. The browser cloud filter was set permissively (70%) so that scenes clearover Stoke but cloudy elsewhere were not hidden from the list.| Season | Date | Notes ||---|---|---|| Summer | 2026-07-09 | Peak vegetation. Chosen from three near-cloudless July candidates. || Autumn | 2025-09-22 | The only date in Sep–Oct 2025 clear over Stoke. Post-harvest — see caveat below. || Winter | 2025-12-26 | Cloudiest season at 53°N; the least-bad usable scene. || Spring | 2026-05-25 | Previously held scene, re-run under current code. |**Autumn caveat.** September is post-harvest, so bare agricultural soil is at itsmost abundant and most confusable with brownfield. Ploughed fields entering thecandidate pool on this date could produce false persistence. This was recordedas a known risk before the runs, not after.

### A note on database hygiene before these runsThe `candidate_sites` table held 794 rows for 2026-05-25 across **eight separateruns**, accumulated over the FND/P0/P1 refactor period. Their counts (218, 218,112, 81, 22, 78 ...) record successive changes to the detection algorithm ratherthan successive observations of the ground.Those rows were archived to `candidate_sites_archive_20260723` and deleted. Thepredicate used was `std_bsi IS NULL`, which selects exactly the runs predatingmigration 004 — an objective test of code generation rather than a judgementabout dates.May 2026 was then re-run under current code and reproduced its previous figuresexactly (90 candidates, 78 after exclusion, 18 register matches), which serves asa reproducibility check on the merged pipeline.All four dates below therefore come from one run each, all from the samecodebase.

## Setup

In [1]:
import sysfrom pathlib import Pathsys.path.insert(0, str(Path.cwd().parent))import pandas as pdfrom src.database_query import get_db_connectionGSS = "E06000021"WINTER = "2025-12-26"     # anchor scene: strictest, fewest candidatesMATCH_M = 50              # centroid proximity for cross-date matchingconn = get_db_connection()print("connected")

SyntaxError: invalid syntax (1199453489.py, line 1)

## 2. Per-date resultsEach row is one pipeline run. `candidate pixels %` is the share of validin-boundary pixels passing the BSI/NDVI gate before clustering.

In [2]:
per_date = pd.read_sql("""    SELECT image_date,           COUNT(*)                                        AS candidates,           COUNT(matched_site_reference)                   AS register_matched,           COUNT(*) - COUNT(matched_site_reference)        AS unregistered,           ROUND(AVG(pixel_count)::numeric, 1)             AS mean_pixels,           ROUND(AVG(bsi_value)::numeric, 4)               AS mean_bsi    FROM candidate_sites    WHERE gss_code = %(gss)s    GROUP BY image_date    ORDER BY image_date""", conn, params={"gss": GSS})per_date["register_recall_pct"] = (    100 * per_date["register_matched"] / per_date["candidates"]).round(1)per_date

SyntaxError: invalid syntax (57845965.py, line 1)

The gate-passing pixel share across the four scenes, taken from the run logs:| Date | Season | Candidate pixels | Sites after size filter | After exclusion ||---|---|---|---|---|| 2026-07-09 | Summer | 2.8% | 292 | 251 || 2025-09-22 | Autumn | 1.7% | 167 | 153 || 2025-12-26 | Winter | 1.0% | 77 | 72 || 2026-05-25 | Spring | 0.8% | 90 | 78 |The gradient is physically coherent: bare-soil signal peaks in July and declinesthrough autumn into winter as illumination falls and ground wets. July exceedingMay was not predicted — the expectation was that summer greening would suppressthe count. The most likely explanation is drought-parched grassland and amenityland reading as bare, which would make summer a **worse** discriminator thanassumed rather than a better one.

## 3. PersistenceA candidate is *persistent* if a candidate from every other date lies within50 m of it.The winter scene is used as the anchor. It is the strictest of the four — only 72candidates survived a late-December acquisition, when even marginal scrub hasdamped down — so a site bare in winter **and** the other three seasons is asstrong a persistence signal as four scenes can produce.**Why not use the stored `prior_date_count` column.** That feature is computed atinsert time and counts only dates already present in the table, so it reflects theorder the runs happened to be executed in rather than true cross-date persistence.The query below is symmetric and order-independent.

In [3]:
persistence = pd.read_sql("""    SELECT dates_present, COUNT(*) AS sites FROM (      SELECT w.id,        (SELECT COUNT(DISTINCT o.image_date)           FROM candidate_sites o          WHERE o.gss_code = w.gss_code            AND ST_DWithin(                  ST_SetSRID(ST_MakePoint(o.utm_x, o.utm_y), 32630),                  ST_SetSRID(ST_MakePoint(w.utm_x, w.utm_y), 32630),                  %(m)s)        ) AS dates_present      FROM candidate_sites w      WHERE w.gss_code = %(gss)s AND w.image_date = %(winter)s    ) t    GROUP BY dates_present    ORDER BY dates_present""", conn, params={"gss": GSS, "winter": WINTER, "m": MATCH_M})persistence

SyntaxError: invalid syntax (4025145694.py, line 1)

**The filter does remove transient ground.** Of 72 winter candidates, 26 appearon the winter date alone — wet ground, temporary works, a single ploughing — andpersistence discards them. Twenty survive all four dates.That is the set the product definition describes. The rest of this notebook askswhat is actually in it.

In [4]:
persistent = pd.read_sql("""    SELECT w.id,           w.utm_x::int  AS utm_x,           w.utm_y::int  AS utm_y,           w.pixel_count,           ROUND((w.pixel_count * 0.04)::numeric, 2) AS hectares,           ROUND(w.bsi_value::numeric, 4)            AS bsi,           ROUND(w.compactness::numeric, 3)          AS compactness,           w.matched_site_reference    FROM candidate_sites w    WHERE w.gss_code = %(gss)s AND w.image_date = %(winter)s      AND (SELECT COUNT(DISTINCT o.image_date)             FROM candidate_sites o            WHERE o.gss_code = w.gss_code              AND ST_DWithin(                    ST_SetSRID(ST_MakePoint(o.utm_x, o.utm_y), 32630),                    ST_SetSRID(ST_MakePoint(w.utm_x, w.utm_y), 32630),                    %(m)s)) = 4    ORDER BY w.pixel_count DESC""", conn, params={"gss": GSS, "winter": WINTER, "m": MATCH_M})n_reg = persistent["matched_site_reference"].notna().sum()print(f"persistent sites: {len(persistent)}  |  register-matched: {n_reg}"      f"  |  unregistered: {len(persistent) - n_reg}")persistent

SyntaxError: invalid syntax (3577049854.py, line 1)

### Two signals visible before any imagery was opened**Register recall collapsed.** Single dates matched the brownfield register at12–23% of candidates. The persistent set matches at 1 of 20 — 5%. If the registeris a reasonable proxy for real brownfield, persistence is moving the candidatepool *away* from brownfield rather than toward it.**Every compactness value is low.** Compactness here is the Polsby–Popper ratio4πA/P²: a circle scores 1.0, a square 0.785. The persistent set ranges from 0.032to 0.321. Every member is long and thin. The largest, at 1.44 ha, scores 0.032 —a ribbon, not a parcel.Both signals pointed the same way before a single site was inspected.

## 4. Manual validationThe unregistered persistent candidates were inspected individually against aerialimagery and labelled per `docs/labelling_protocol.md`. `sellable` is reserved fora discrete parcel, with boundaries, showing no active use, that a developer couldplausibly buy. Everything else takes a false-positive class.Labels are loaded from CSV rather than hardcoded, so this notebook can be re-runas further sites are labelled.**Caveat on aerial imagery.** Google's imagery may predate the satellite scenesused here. Where a site's status was ambiguous because of that mismatch, it wasleft unlabelled rather than guessed; unlabelled rows are excluded from thedenominator.

In [5]:
labels = pd.read_csv("../outputs/persistent_labels.csv")labelled = labels[labels["label"].notna() & (labels["label"].astype(str).str.strip() != "")]print(f"labelled: {len(labelled)} of {len(labels)} unregistered persistent candidates")labelled[["candidate_id", "hectares", "compactness", "label", "site_name"]]

SyntaxError: invalid syntax (4062138652.py, line 1)

In [6]:
breakdown = labelled["label"].value_counts().rename_axis("label").reset_index(name="sites")sellable = (labelled["label"] == "sellable").sum()print(f"sellable: {sellable} / {len(labelled)}"      f"  =  precision {100 * sellable / len(labelled):.1f}%")breakdown

SyntaxError: invalid syntax (1401925151.py, line 1)

### What the sites turned out to beEvery inspected candidate resolved to an active, occupied site:| id | Site | Class ||---|---|---|| 1362 | Royal Stoke University Hospital | `active_institutional` || 1361 | Mossfield Road estate — D&G Bus, IAE | `active_industrial` || 1320 | Ormiston Horizon Academy | `active_institutional` || 1367 | Park Hall Business Village | `active_industrial` || 1371 | Fenton — Victoria Industrial Complex | `active_industrial` || 1332 | Scotia Business Park, Tunstall | `active_industrial` |In every case the detected footprint is **roofs and hardstanding** — warehouseand unit roofs, bus and lorry yards, hospital and school service areas, carparking — not land.This is consistent with the earlier 19/19 labelling pilot on single-datecandidates, which found no sellable sites either. Persistence did not change thecharacter of the false positives; it selected for a particular subtype of them.

## 5. Finding> **The hypothesis is not supported.** Requiring bareness across four seasons> does remove transient bare ground, but the set it leaves is dominated by> *active* commercial and institutional sites rather than derelict land.The mechanism is straightforward once stated:**Persistence selects for permanence of use, not absence of use.** A warehouseroof, a bus depot yard and a hospital car park are bare in every season becausethey are in continuous use and are maintained that way. Genuinely derelict land,by contrast, is frequently colonised by vegetation and may read as *vegetated* onone or more dates — so the filter can actively select against the target.**The root cause is upstream of persistence.** BSI and NDVI separate vegetationfrom non-vegetation. They cannot separate bare ground from hard surface: a tarmacyard and a demolished plot are close to indistinguishable in both indices.Persistence then compounds the error, because hard surfaces are the most reliablynon-vegetated things in a city.Notebook 04 anticipated part of this. It warned that threshold-only detectionwould find currently-bare land and little else, and recorded that the multi-bandapproach was abandoned. That warning was borne out by low register recall onsingle dates; this notebook shows the same limitation is not fixed by addingdates.

### Caveats on this findingStated plainly, so the conclusion is not read as stronger than the evidence.**The cross-date join is loose.** Candidates are matched between dates bycentroid proximity within 50 m. A large irregular site whose centroid shiftsbetween dates could fail to match itself; two distinct adjacent sites could matcheach other. Candidate geometry is now stored (FND-3), so a polygon-overlap joinwould be tighter and should replace this before any figure is published.**The labelled sample is partial.** The sites inspected are drawn from the largerend of the persistent set. Smaller candidates (0.2–0.36 ha) are a differentpopulation and could plausibly contain infill plots or demolished terraces. Theprecision figure computed above reflects only what has been labelled.**`min_pixels = 5` remains uncalibrated** (issue #89). The size threshold shapingthe candidate pool is provisional, so the composition of that pool is provisionalwith it.**Four dates is few.** Only one scene per season was usable, and only one autumndate in the entire two-month window was clear over Stoke. A wetter or drier yearwould produce a different set.

## 6. What follows**Building-footprint subtraction (issue #96).** The specific fix indicated by theevidence. OSM carries building polygons for Stoke; a candidate whose footprintlies mostly over mapped buildings is a roof, not land, and can be dropped. Thistargets the actual mechanism. Note that it is sharper than a compactnessthreshold for this purpose — a rectangular warehouse roof is perfectly compact —though compactness may still help against linear infrastructure.Building was deliberately excluded from the hard exclusion classes in FND-4, onthe reasoning that derelict buildings are themselves brownfield. That reasoningstill holds. The distinction required is **active versus abandoned**, whichland-use class alone does not supply; occupancy signals are needed.**This bears directly on the Notebook 07 architecture fork.** Notebook 07 leftopen a choice between (A) re-ranking threshold-gated candidates and (B) replacingthe gate with a learned detector. If the candidate pool is dominated by activeroofs and yards, fork A re-ranks the wrong population, and no classifier trainedon it can surface land that never entered the pool. This finding is evidence forfork B, or at minimum for fixing the gate before training anything on its output.**Sequence from here:** implement building subtraction, re-run the four scenes,re-derive the persistent set, and re-label. If the set changes character, thepersistence hypothesis is worth re-testing on a cleaner pool; if it does not, thegate itself needs replacing.

In [7]:
conn.close()print("closed")

SyntaxError: invalid syntax (3239514285.py, line 1)